# ML WAF — data, features, model

Exploration notebook for the SQLi / XSS detector.

The runtime code lives in `src/mlwaf/` — this notebook only looks at it.
Run `python -m mlwaf.download && python -m mlwaf.dataset && python -m mlwaf.train` first.

**Why the old approach failed.** v0 labelled a request `bad` when any of six
counters (quotes, dashes, parens, spaces, SQL keywords) was non-zero, then trained
k-means on those same six counters. The label was a function of the features, so
the model could only rediscover a rule already written by hand — and it recovered
it imperfectly, catching 52% of attacks. Here the labels come from the ECML/PKDD
2007 challenge, which tags each request with its actual attack type.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from mlwaf.features import NUMERIC_COLS, build_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)

labelled = pd.read_parquet("../data/processed/ecml_labelled.parquet")
unseen = pd.read_parquet("../data/processed/ecml_unseen.parquet")
csic = pd.read_parquet("../data/processed/csic.parquet")
labelled["label"].value_counts()

## 1. What a request looks like after normalisation

In [ ]:
for label in ["benign", "sqli", "xss"]:
    row = labelled[labelled["label"] == label].iloc[0]
    print(f"--- {label}")
    print(row["text"][:240])
    print()

## 2. The decode chain

Payloads arrive wrapped in encoding layers. Counting characters on the raw string
misses them entirely; we peel first and record how many rounds it took.

In [ ]:
from mlwaf.decode import decode

samples = [
    "%2527%2520OR%25201%3D1",
    "&#60;script&#62;alert(1)&#60;/script&#62;",
    r"\u003cimg src=x onerror=alert(1)\u003e",
    "/products/search",
]
for s in samples:
    text, rounds = decode(s)
    print(f"rounds={rounds}  {s[:44]:<46} -> {text}")

In [ ]:
ax = sns.boxplot(data=labelled, x="label", y="decode_depth", order=["benign", "sqli", "xss"])
ax.set_title("Encoding layers per request")
plt.show()

## 3. Why the six counters were never enough

The v0 features could not separate a surname from an auth bypass — both are
"one single quote". Character n-grams see the pattern, not the tally.

In [ ]:
from mlwaf.features import numeric_features

collision = pd.DataFrame([
    {"text": "get /users q=o'brien", "query": "q=o'brien", "path": "/users", "decode_depth": 0},
    {"text": "get /login q=admin' or 1=1", "query": "q=admin' or 1=1", "path": "/login", "decode_depth": 0},
])
numeric_features(collision)[["single_q", "double_q", "dashes", "parens", "sql_keywords"]]

In [ ]:
feat = numeric_features(labelled)
feat["label"] = labelled["label"].values
cols = ["single_q", "sql_keywords", "angle_open", "tags", "event_handlers", "entropy"]
feat.groupby("label")[cols].mean().round(2)

## 4. Results

In [ ]:
metrics = json.loads(Path("../reports/metrics.json").read_text())

rows = []
for name in ["rule_baseline", "logreg", "lightgbm"]:
    m = metrics[name]
    at = m["at_fpr"]["0.001"]
    rows.append({
        "model": name,
        "macro_f1": m["macro_f1"],
        "pr_auc": m["binary_pr_auc"],
        "sqli_recall": m["per_class"]["sqli"]["recall"],
        "xss_recall": m["per_class"]["xss"]["recall"],
        "recall@0.1%fpr": at["recall"],
        "ms/req": m["predict_ms_per_request"],
    })
pd.DataFrame(rows).set_index("model")

In [ ]:
import numpy as np

final = metrics["FINAL_TEST"]
cm = np.array(final["confusion_matrix"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["benign", "sqli", "xss"], yticklabels=["benign", "sqli", "xss"])
plt.xlabel("predicted"); plt.ylabel("actual"); plt.title("Held-out test set")
plt.show()
final["generalisation"]

## 5. Which features the model actually uses

In [ ]:
bundle = joblib.load("../models/model.joblib")
pipe = bundle["pipeline"]

names = pipe.named_steps["features"].get_feature_names_out()
imp = pipe.named_steps["clf"].feature_importances_
top = pd.Series(imp, index=names).sort_values(ascending=False).head(25)
top.sort_values().plot.barh(figsize=(7, 8), title="Top 25 features")
plt.show()

In [ ]:
# Try it on payloads that appear nowhere in the corpus.
probes = [
    ("benign",  "GET", "/products/search", "q=running+shoes", ""),
    ("benign",  "POST", "/account/update", "", "name=O'Brien&city=Cork"),
    ("sqli",    "GET", "/item", "id=1%27+UNION+SELECT+username%2Cpassword+FROM+users--", ""),
    ("sqli",    "GET", "/item", "id=1%27+AND+SLEEP%285%29--", ""),
    ("xss",     "GET", "/search", "q=%3Cscript%3Ealert%281%29%3C%2Fscript%3E", ""),
    ("xss",     "GET", "/search", "q=%3Cimg+src%3Dx+onerror%3Dalert%281%29%3E", ""),
]
from mlwaf.decode import request_text

rows = []
for expected, method, path, query, body in probes:
    text, depth = request_text(method, path, query, body)
    rows.append({"text": text, "query": query, "path": path, "decode_depth": depth,
                 "expected": expected})
probe_df = pd.DataFrame(rows)
proba = pipe.predict_proba(build_matrix(probe_df))
probe_df["predicted"] = pipe.predict(build_matrix(probe_df))
probe_df["attack_score"] = (1 - proba[:, 0]).round(4)
probe_df["blocked"] = probe_df["attack_score"] >= bundle["threshold"]
probe_df[["expected", "predicted", "attack_score", "blocked"]]